# PlantCare — Notebook 03: image → advisoryThe stage that connects the other two. Notebook 01 classifies a leaf; notebook 02 builtthe knowledge base. Nothing yet joins them, and **nothing uses the severity number atall** — that is what this notebook is for.```leaf image  -> preprocessing (notebook 01: deblur gate + CLAHE/gamma)  -> EfficientNet-B0 -> class + confidence  -> OOD screens (leaf presence / Mahalanobis / confidence gate)   -> may REFUSE  -> lesion segmentation -> severity %  -> band  -> evidence bundle: 5 blocks for the class + symptoms of look-alike diseases  -> REASONING PROMPT -> Groq -> audit -> extractive fallback  -> advisory```## Do you need it?The old notebook 03 is now redundant — notebook 02 absorbed tiered retrieval, generationand the audit. But notebook 02 produces **13 class-level reports**, one per disease, thatknow nothing about any particular leaf. This notebook produces a **per-image** advisorythat knows the confidence, the severity, and what the disease could be confused with.The severity work from notebook 01 is currently orphaned. Here it drives the urgencylevel.## What is new here, and the one thing that could go wrongThe LLM **has not seen the image**. If the prompt lets it drift into "the leaf showsbrown concentric rings", that sentence is invented — no source and no observation behindit — and it would read as the most authoritative line in the report. The prompt bansimage claims outright and the audit checks for them mechanically.

## Datasets to attach**Add data** in the right-hand panel, three times:| # | Dataset | Where it comes from | Used for ||---|---|---|---|| 1 | `Tomato_Grape_Crop_Dataset` | you already have it | test leaf images || 2 | **notebook 01 artifacts** | upload notebook 01's output zip as a new Kaggle dataset | `best_model.pt`, `preprocessing.py`, `severity.py`, `run_config.json`, OOD stats || 3 | **notebook 02 artifacts** | upload the `kb_export` folder as a new Kaggle dataset | `kb_chunks.json`, `retriever.py`, `embeddings.npy`, `chunks.csv` |To make 2 and 3: Kaggle → **Datasets → New Dataset → Upload**, drop the files in, name it,and it appears under **Add data → Your Datasets**.Also needed: **Settings → Internet → On**, and `GROQ_API_KEY` in **Add-ons → Secrets**.If dataset 2 is missing the notebook still runs — it drops to `MANUAL_CLASS` mode whereyou supply the class and severity yourself, so you can test the reasoning layer beforewiring up the model.

## 0. Setup

In [ ]:
import sys, subprocessdef pipi(*p):    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *p], check=False)try:    import pymilvus, milvus_lite          # noqaexcept ImportError:    pipi("-U", "pymilvus", "milvus-lite")try:    import sentence_transformers          # noqaexcept ImportError:    pipi("sentence-transformers")import os, re, io, json, math, time, glob, shutil, textwrap, importlib.utilfrom pathlib import Pathfrom collections import Counter, defaultdictimport numpy as np, pandas as pd, requestsfrom pymilvus import MilvusClient, DataTypeimport torchprint("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 1. Configuration and dataset discovery

In [ ]:
INPUT = Path("/kaggle/input")WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")OUT   = WORK / "advisories"; OUT.mkdir(parents=True, exist_ok=True)def find_one(pattern, what):    hits = sorted(INPUT.rglob(pattern))    if not hits:        print(f"  NOT FOUND: {what}  (pattern {pattern})")        return None    if len(hits) > 1:        print(f"  {what}: {len(hits)} matches, using {hits[0]}")    return hits[0]print("locating artifacts under /kaggle/input ...")KB_JSON         = find_one("kb_chunks.json",    "notebook 02 knowledge base")EMB_NPY         = find_one("embeddings.npy",    "notebook 02 embeddings")CKPT            = find_one("best_cnn.pt",       "notebook 01 checkpoint")RUN_CONFIG_JSON = find_one("run_config.json",   "notebook 01 run config")CLASS_INDEX_JSON= find_one("class_index.json",  "notebook 01 class index")OOD_NPZ         = find_one("ood_gaussian.npz",  "notebook 01 OOD gaussian")PREPROC_PY      = find_one("preprocessing.py",  "notebook 01 preprocessing module")SEVERITY_PY     = find_one("severity.py",       "notebook 01 severity module")IMAGE_ROOTS = [p for p in INPUT.iterdir()               if p.is_dir() and any(p.rglob("*.JPG")) or any(p.rglob("*.jpg"))]print("\nimage roots:", [str(p.name) for p in IMAGE_ROOTS] or "none")MILVUS_DB  = str(WORK / "plantcare_kb.db")COLLECTION = "plantcare_kb"# ---------------------------------------------------------------- retrievalST_MODEL     = "BAAI/bge-base-en-v1.5"EMBED_DIM    = 768QUERY_PREFIX = "Represent this sentence for searching relevant passages: "TOP_K_DENSE, TOP_K_SPARSE, RRF_K, BLOCK_K = 20, 20, 60, 3DIFF_K       = 2      # look-alike diseases to contrast againstDIFF_PASSAGES= 2      # symptom passages per look-alikeBM25_K1, BM25_B = 1.5, 0.75# ---------------------------------------------------------------- severity# NOT redefined here. Notebook 01 owns the thresholds (HEALTHY_SEVERITY 5 / MILD_MAX 15# / MODERATE_MAX 35) and its severity_level() is imported from the exported severity.py,# so the bands cannot drift between the two notebooks.## What IS defined here is the map from notebook 01's band strings to an urgency phrase.# It lives in this cell, not the CNN cell, because the prompt cell runs first and needs it.BAND_URGENCY = {    "Healthy":     "none - monitoring only",    "Mild":        "low - act within the week",    "Moderate":    "moderate - act within 2-3 days",    "High":        "high - act immediately",    "Unavailable": "unknown - the leaf mask failed, treat the severity as unmeasured",}# The two models can disagree, and one direction of disagreement is dangerous.## The lesion mask is chroma/darkness based. It finds brown necrotic spots well and# DIFFUSE symptoms badly — leaf mold's pale greenish-yellow blotches, early TYLCV# curling and chlorosis, spider-mite stippling. A real leaf-mold leaf can measure 0.1%# and land in the "Healthy" band while the CNN is 95% sure it is diseased.## Taking the band at face value there would print "urgency: none - monitoring only" on a# confirmed disease. So when the CNN names a disease and the mask says Healthy, the# severity is treated as UNMEASURED rather than as good news. The classifier decides# whether there is disease; the mask only decides how much.def resolve_urgency(case):    cls, band = case["disease"], case["severity_level"]    diseased = cls not in HEALTHY_CLASSES    if diseased and band in ("Healthy", "Unavailable"):        return ("low - act within the week (severity could not be measured reliably)",                f"The lesion mask measured {case['severity_pct']:.1f}% of leaf area, which "                f"would normally read as healthy, but the classifier identifies "                f"{cls.replace('_', ' ')} with {case['confidence']:.0%} confidence. This "                f"disease can present as diffuse discolouration that the mask does not "                f"capture. Treat the percentage as unmeasured, not as an all-clear.")    return BAND_URGENCY[band], None# ---------------------------------------------------------------- HuggingFace# Optional. Anonymous downloads work but are rate-limited and slower; on a shared Kaggle# IP a burst of anonymous pulls can return 429 and fail the embedding cell outright.HF_TOKEN_SECRET = "HF_TOKEN"# ---------------------------------------------------------------- LLMPROVIDER  = "groq"BASE_URL  = "https://api.groq.com/openai/v1"SECRET    = "GROQ_API_KEY"PREFER    = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b",             "qwen3-32b", "llama-3.1-8b-instant"]LLM_TEMP  = 0.15REQ_PAUSE = 6.0MAX_RETRIES = 3# ---------------------------------------------------------------- fallback mode# Used when the notebook 01 artifacts are absent: supply the case by hand so the# reasoning + audit layer can still be exercised.MANUAL_CLASS    = "tomato_early_blight"MANUAL_CONF     = 0.97MANUAL_SEVERITY = 18.4print("\nMilvus DB:", MILVUS_DB)

## 2. Preflight

In [ ]:
# ---- HuggingFace token (optional) --------------------------------------HF_TOKEN = ""try:    from kaggle_secrets import UserSecretsClient    HF_TOKEN = UserSecretsClient().get_secret(HF_TOKEN_SECRET)except Exception:    HF_TOKEN = os.environ.get("HF_TOKEN", "")if HF_TOKEN:    os.environ["HF_TOKEN"] = HF_TOKEN    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN     # older huggingface_hub reads this    print("HF    : token set - authenticated downloads")else:    print("HF    : no token (anonymous downloads work, just rate-limited)")API_KEY, LLM_OK, LLM_MODEL = "", False, Nonetry:    from kaggle_secrets import UserSecretsClient    API_KEY = UserSecretsClient().get_secret(SECRET)except Exception:    API_KEY = os.environ.get(SECRET, "")if API_KEY:    try:        r = requests.get(f"{BASE_URL}/models", timeout=30,                         headers={"Authorization": f"Bearer {API_KEY}"})        r.raise_for_status()        live = [m["id"] for m in r.json().get("data", [])]        LLM_MODEL = next((m for m in PREFER if m in live), None) or next(            (m for m in live if not re.search(r"whisper|guard|tts|embed|rerank", m, re.I)),            None)        LLM_OK = LLM_MODEL is not None        print(f"LLM   : {LLM_MODEL}  ({len(live)} models live)")    except Exception as e:        print("LLM   : API check failed:", type(e).__name__, str(e)[:150])else:    print(f"LLM   : no {SECRET} in Secrets -> advisories will be EXTRACTIVE")CNN_OK = bool(CKPT and RUN_CONFIG_JSON and SEVERITY_PY)print("CNN   :", "artifacts found" if CNN_OK else      f"incomplete -> MANUAL_CLASS mode ({MANUAL_CLASS})")assert KB_JSON, "notebook 02 kb_chunks.json is required - attach that dataset"print("KB    :", KB_JSON)

## 3. Rebuild the vector store from `kb_chunks.json`Faster and more reliable than shipping the `.db` between notebooks: 172 chunks insert inabout a second, and it sidesteps the Milvus Lite quirk where a reopened collection comesback in the `released` state.

In [ ]:
payload  = json.loads(Path(KB_JSON).read_text(encoding="utf-8"))manifest = payload["manifest"]chunks   = payload["chunks"]CLASSES         = manifest["classes"]HEALTHY_CLASSES = set(manifest["healthy_classes"])COVERAGE        = manifest["coverage"]print(f"{len(chunks)} chunks | {manifest['n_documents']} documents "      f"| built by {manifest['built_by']}")print("verdicts:", dict(Counter(v["verdict"] for v in COVERAGE.values())))_ST = Nonedef embed(texts, is_query=False, batch=32):    global _ST    if _ST is None:        import logging, warnings        # bge-base ships a legacy `embeddings.position_ids` buffer that current        # transformers no longer registers. It is a non-parameter buffer, so the        # "UNEXPECTED" line in the load report is cosmetic - the weights are complete.        logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)        warnings.filterwarnings("ignore", message=".*position_ids.*")        from sentence_transformers import SentenceTransformer        dev = "cuda" if torch.cuda.is_available() else "cpu"        print(f"loading {ST_MODEL} on {dev} ...")        _ST = SentenceTransformer(ST_MODEL, device=dev,                                  token=HF_TOKEN or None)    if is_query:        texts = [QUERY_PREFIX + t for t in texts]    v = _ST.encode(texts, batch_size=batch, normalize_embeddings=True,                   show_progress_bar=len(texts) > 64)    return np.asarray(v, dtype=np.float32)# reuse notebook 02's vectors when the shapes line up; recompute otherwisedense = Noneif EMB_NPY:    cand = np.load(EMB_NPY)    if cand.shape == (len(chunks), EMBED_DIM):        dense = cand.astype(np.float32)        print("reused embeddings.npy", dense.shape)    else:        print(f"embeddings.npy is {cand.shape}, expected {(len(chunks), EMBED_DIM)} "              "- recomputing")if dense is None:    dense = embed([c["text"] for c in chunks])assert np.allclose(np.linalg.norm(dense, axis=1), 1.0, atol=1e-4), "not unit-norm"

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")def tokenize(s):    return TOKEN_RE.findall(s.lower())docs_tok = [tokenize(c["search_text"]) for c in chunks]N, avgdl = len(docs_tok), sum(len(d) for d in docs_tok) / len(docs_tok)df = Counter()for d in docs_tok:    df.update(set(d))vocab = {t: i for i, t in enumerate(sorted(df))}idf   = {t: math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5)) for t in df}def doc_sparse(tok):    tf, dl = Counter(tok), len(tok)    return {vocab[t]: float(f * (BM25_K1 + 1) /                            (f + BM25_K1 * (1 - BM25_B + BM25_B * dl / avgdl)))            for t, f in tf.items()}def query_sparse(text):    return {vocab[t]: float(idf[t]) for t in set(tokenize(text)) if t in vocab}sparse_docs = [doc_sparse(d) for d in docs_tok]# Milvus Lite writes the "db" as a DIRECTORY (collections/, databases/, LOCK), not a# single file. os.remove() raises IsADirectoryError on it, os.path.getsize() reports the# 4 KB inode instead of the real contents, and any p.is_file() filter skips it entirely.def reset_path(p):    if os.path.isdir(p):        shutil.rmtree(p)    elif os.path.exists(p):        os.remove(p)def path_size(p):    if os.path.isfile(p):        return os.path.getsize(p)    if os.path.isdir(p):        return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())    return 0reset_path(MILVUS_DB)client = MilvusClient(uri=MILVUS_DB)s = client.create_schema(auto_id=False, enable_dynamic_field=False)s.add_field("id", DataType.INT64, is_primary=True)s.add_field("text", DataType.VARCHAR, max_length=4000)s.add_field("disease", DataType.VARCHAR, max_length=64)s.add_field("crop", DataType.VARCHAR, max_length=32)s.add_field("section_type", DataType.VARCHAR, max_length=32)s.add_field("doc", DataType.VARCHAR, max_length=256)s.add_field("page", DataType.INT64)s.add_field("dense", DataType.FLOAT_VECTOR, dim=EMBED_DIM)s.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)ix = client.prepare_index_params()ix.add_index(field_name="dense", index_type="FLAT", metric_type="IP")ix.add_index(field_name="sparse", index_type="SPARSE_INVERTED_INDEX", metric_type="IP")client.create_collection(COLLECTION, schema=s, index_params=ix)client.insert(COLLECTION, [{    "id": int(c["id"]), "text": c["text"][:4000], "disease": c["disease"],    "crop": c["crop"], "section_type": c["section_type"],    "doc": c["doc"][:256], "page": int(c["page"]),    "dense": dense[i].tolist(), "sparse": sparse_docs[i],} for i, c in enumerate(chunks)])client.flush(COLLECTION)client.load_collection(COLLECTION)print("Milvus Lite ready:", client.get_collection_stats(COLLECTION))by_id = {c["id"]: c for c in chunks}def rrf(*lists, k=RRF_K):    score = defaultdict(float)    for lst in lists:        for rank, i in enumerate(lst, start=1):            score[i] += 1.0 / (k + rank)    return sorted(score.items(), key=lambda x: -x[1])def hybrid_search(query, expr=None, k=BLOCK_K):    qv = embed([query], is_query=True)[0]    d = client.search(COLLECTION, data=[qv.tolist()], anns_field="dense",                      limit=TOP_K_DENSE, filter=expr or "", output_fields=["id"],                      search_params={"metric_type": "IP"})[0]    qs = query_sparse(query)    sp = client.search(COLLECTION, data=[qs], anns_field="sparse", limit=TOP_K_SPARSE,                       filter=expr or "", output_fields=["id"],                       search_params={"metric_type": "IP"})[0] if qs else []    fused = rrf([h["id"] for h in d], [h["id"] for h in sp])[:k]    return [dict(by_id[i], _rrf=sc) for i, sc in fused]print("retrieval ready")

## 4. Which diseases look alike — derived, not hand-writtenA curated confusion list would be my opinion. Instead we compute a **symptom centroid**per class (the mean of its `symptom` chunk embeddings) and take the nearest other classesin the same crop by cosine similarity.This is defensible because it measures what the corpus actually says: if two diseases aredescribed with similar symptom language, a farmer looking at a leaf will have the sametrouble the embedding does. It also updates automatically when documents change.

In [ ]:
sym_idx = defaultdict(list)for i, c in enumerate(chunks):    if c["section_type"] == "symptom" and c["disease"] not in HEALTHY_CLASSES:        sym_idx[c["disease"]].append(i)centroid = {}for cls, idxs in sym_idx.items():    v = dense[idxs].mean(axis=0)    centroid[cls] = v / (np.linalg.norm(v) + 1e-9)def look_alikes(cls, k=DIFF_K):    if cls not in centroid:        return []    crop = cls.split("_")[0]    sims = [(float(centroid[cls] @ centroid[o]), o) for o in centroid            if o != cls and o.split("_")[0] == crop]    sims.sort(reverse=True)    return [(o, s) for s, o in sims[:k]]print("symptom-similarity neighbours within each crop\n")for cls in sorted(centroid):    la = ", ".join(f"{o} ({s:.3f})" for o, s in look_alikes(cls))    print(f"  {cls:32s} -> {la}")

## 5. Evidence bundle

In [ ]:
SECTION_QUERIES = {    "identity":   ("pathogen",     "{d} causal pathogen scientific name organism"),    "symptoms":   ("symptom",      "{d} symptoms on leaves early and advanced signs"),    "spread":     ("transmission", "{d} transmission how it spreads overwintering conditions"),    "treatment":  ("treatment",    "{d} treatment chemical biological control fungicide"),    "prevention": ("prevention",   "{d} prevention cultural practices sanitation resistant"),}def pretty(cls):    return cls.replace("_", " ")def tier_for(cls):    if cls in HEALTHY_CLASSES:        return "healthy"    if COVERAGE.get(cls, {}).get("verdict") in ("OK", "THIN"):        return "A"    crop = cls.split("_")[0]    if any(c != cls and c not in HEALTHY_CLASSES and c.startswith(crop + "_")           and COVERAGE[c]["verdict"] in ("OK", "THIN") for c in COVERAGE):        return "B"    return "C"def build_evidence(cls):    """-> (target_blocks, differential, caveats)"""    tier = tier_for(cls)    caveats = []    if tier == "C":        return {}, {}, ["No source document covers this class."]    if tier == "A":        expr = f'disease == "{cls}"'    else:        crop = cls.split("_")[0]        expr = f'crop == "{crop}" and disease != "{cls}"'        caveats.append(            f"No document covers {pretty(cls)}. The passages below describe other {crop} "            f"diseases and are general context only. Treatment guidance is withheld "            f"because it would not be specific to this disease.")    blocks, seen = {}, set()    for name, (stype, tmpl) in SECTION_QUERIES.items():        if tier == "B" and stype == "treatment":            continue                                  # safety rule from notebook 02        q = tmpl.format(d=pretty(cls))        hits = hybrid_search(q, expr=f'{expr} and section_type == "{stype}"') \               or hybrid_search(q, expr=expr)        keep = [h for h in hits if h["id"] not in seen]        seen.update(h["id"] for h in keep)        if keep:            blocks[name] = keep    differential = {}    if tier == "A":        for other, _sim in look_alikes(cls):            hits = hybrid_search(f"{pretty(other)} symptoms on leaves distinguishing signs",                                 expr=f'disease == "{other}" and section_type == "symptom"',                                 k=DIFF_PASSAGES)            keep = [h for h in hits if h["id"] not in seen]            seen.update(h["id"] for h in keep)            if keep:                differential[other] = keep    return blocks, differential, caveatstb, dfx, cav = build_evidence("tomato_early_blight")print("target blocks :", {k: len(v) for k, v in tb.items()})print("look-alikes   :", {k: len(v) for k, v in dfx.items()})

## 6. The reasoning prompt templateThis is the part that decides whether the output is trustworthy, so it is worth readingline by line rather than skimming.### The core problem it solvesThe model **cannot see the image**. Everything visual it might say is either (a) a generalstatement about the disease, which a source can support, or (b) a claim about thisparticular leaf, which nothing supports. Type (b) is the dangerous one because it readsas the most authoritative sentence in the whole report — *"the leaf shows brown concentricrings with a yellow halo"* sounds like observation and is pure invention.So rule 1 bans image claims, and the audit in the next cell greps for them.### Why the diagnosis and severity are stated as fixed factsThey come from measurement — a CNN and a segmentation mask. If the prompt invites themodel to weigh them, a fluent 70B model will sometimes decide the classifier was wrong,and it will be wrong about that far more often than the classifier is. Rule 5 removes theoption: explain and act on the diagnosis, don't relitigate it.### Where the actual reasoning happensThe **"What else it could look like"** section. That is a real differential, grounded inretrieved symptom passages from the two nearest-by-symptom diseases in the same crop. Themodel has to name a *cited* difference. It is the one place where combining passages addssomething neither passage contains alone — and it is exactly the question a farmer holdinga spotted leaf actually has.The escape hatch matters too: if the sources give no basis to separate two diseases, themodel is told to say so rather than manufacture a distinction.### Severity drives urgency, but code sets the band`SEVERITY_URGENCY` maps band → urgency in Python, and the prompt receives the result as agiven. The model chooses which cited actions fit that urgency; it never decides that 18%is "moderate".

In [ ]:
REASONING_SYSTEM = """You are an agricultural extension advisor writing a field advisory.You are one stage in a pipeline. An image classifier has already identified the diseaseand a segmentation model has already measured how much leaf area is affected. You did NOTsee the image and you cannot see it now.ABSOLUTE RULES1. NEVER describe the image. You have not seen it. Do not write "the image shows",   "the leaf in the photo", "visible in this sample", or anything similar. Write about   what the disease TYPICALLY does, citing a source: "Early blight typically begins on   the oldest lowest leaves [S3]."2. Use ONLY the numbered SOURCES. No outside knowledge. If the sources do not cover   something, write exactly: "The sources do not state this."3. NEVER invent a chemical name, a dose, a rate, a concentration, a spray interval or a   temperature. If a number is not in the sources, it does not go in your answer.4. Every sentence ends with one or more citation markers: [S1], or [S2][S5].5. The DIAGNOSIS and the SEVERITY BAND are given to you as facts. Do not overturn them,   re-diagnose, or dispute the percentage. Your job is to explain and act on them.6. Reason only about the EVIDENCE. Where you compare the diagnosis against a look-alike   disease, ground every contrast in a cited source passage from the sources given.Write in plain language a farmer can act on. Short sentences. No preamble, no sign-off."""REASONING_TEMPLATE = """CASE  Crop              : {crop}  Diagnosis         : {disease_pretty}  Classifier confidence : {confidence:.1%}  Leaf area affected: {severity_pct:.1f}%  -> band: {severity_band}  Urgency (from band, fixed by the pipeline): {urgency}SOURCES FOR {disease_upper}{target_sources}SYMPTOM PASSAGES FOR LOOK-ALIKE DISEASES(these are the {n_diff} diseases in the same crop whose described symptoms are closest tothe diagnosis; they are given so you can contrast, NOT so you can re-diagnose){differential_sources}{caveat_block}TASKWrite the advisory using exactly these seven headings, in this order:## What this disease isOne or two sentences: the pathogen and what kind of organism it is. Cite.## Why this diagnosis fitsDescribe the symptoms this disease TYPICALLY produces, from the sources. Do not claim tohave observed them. Cite every sentence.## What else it could look likeFor each look-alike disease listed above, give ONE sentence naming a cited symptomdifference that separates it from {disease_pretty}. Format each as:  - **<Other disease>:** <the difference> [S#]If the sources give you no basis to separate two diseases, say so for that diseaseinstead of inventing a distinction.## How it spreadsCite. Two sentences maximum.## What to do nowActions justified by the {urgency} urgency level. Draw from the treatment and preventionsources. Every action cited. If the sources give no treatment, say so — do not substitutea general recommendation.## What to keep doingPreventive and cultural practices for the rest of the season. Cite.## Limits of this adviceTwo sentences. State plainly that the diagnosis came from an image classifier and not alaboratory test, and that chemical choices must follow local registration and label rates.No citation needed for this heading only."""def build_reasoning_prompt(case, target_blocks, differential, caveats):    """case: dict from classify_and_measure(). Returns (user_prompt, meta)."""    meta, tgt_lines = [], []    for block_name, hits in target_blocks.items():        for h in hits:            meta.append(h)            tgt_lines.append(f"[S{len(meta)}] ({block_name} | {h['doc']} p.{h['page']})\n"                             f"      {h['text']}")    diff_lines = []    for other_cls, hits in differential.items():        diff_lines.append(f"  --- {pretty(other_cls)} ---")        for h in hits:            meta.append(h)            diff_lines.append(f"[S{len(meta)}] ({h['doc']} p.{h['page']})\n"                              f"      {h['text']}")    urgency_text, conflict = resolve_urgency(case)    notes = list(caveats) + ([conflict] if conflict else [])    cav = ("\nIMPORTANT CONTEXT\n\n  " + "\n  ".join(notes) + "\n") if notes else ""    user = REASONING_TEMPLATE.format(        crop=case["crop"],        disease_pretty=pretty(case["disease"]),        disease_upper=pretty(case["disease"]).upper(),        confidence=case["confidence"],        severity_pct=case["severity_pct"],        severity_band=case["severity_level"],        urgency=urgency_text,        target_sources="\n\n".join(tgt_lines),        n_diff=len(differential),        differential_sources="\n\n".join(diff_lines) if diff_lines                             else "  (none - no other disease in this crop is covered)",        caveat_block=cav,    )    return user, meta# ---------------------------------------------------------------- preview_case = {"disease": "tomato_early_blight", "crop": "tomato",         "confidence": 0.973, "severity_pct": 18.4, "severity_level": "Moderate"}_tb, _df, _cav = build_evidence(_case["disease"])_p, _meta = build_reasoning_prompt(_case, _tb, _df, _cav)print(f"prompt: {len(_p)} chars, {len(_meta)} sources "      f"({sum(len(v) for v in _tb.values())} target + {sum(len(v) for v in _df.values())} differential)")print("=" * 84)print(_p[:2600])print("...")

## 7. The auditNotebook 02's audit plus two checks this notebook needs:* **image-claim detection** — the failure mode described above;* **differential integrity** — every disease named in the look-alike section must be one  we actually retrieved passages for, so the model can't contrast against a disease it  invented.

In [ ]:
NUM_UNIT_RE = re.compile(    r"\b\d+(?:[.,]\d+)?\s*(?:-\s*\d+(?:[.,]\d+)?\s*)?"    r"(?:%|g|kg|mg|ml|l|litre|liter|lb|oz|ha|acre|day|days|week|weeks|hour|hours|year|years|"    r"ppm|mesh|cm|mm|inch|inches|°c|°f)\b", re.I)CHEM_RE = re.compile(    r"\b(mancozeb|chlorothalonil|copper|captan|myclobutanil|azoxystrobin|difenoconazole|"    r"tebuconazole|propiconazole|streptomycin|bordeaux|sulfur|sulphur|abamectin|spinosad|"    r"imidacloprid|spiromesifen|neem|bacillus|trichoderma|maneb|ziram|fosetyl|mefenoxam|"    r"metalaxyl|acibenzolar|actigard|phytoseiulus)\b", re.I)IMAGE_CLAIM_RE = re.compile(    r"\b(the|this|your)\s+(image|photo|photograph|picture|sample|specimen)\b"    r"|\bin the (image|photo|picture)\b"    r"|\b(i|we) can see\b|\bas (seen|shown|visible) (in|on)\b"    r"|\bthe leaf (shown|pictured|in)\b|\bvisible in\b", re.I)def _norm(s):    return re.sub(r"[\s,]+", " ", s.lower()).strip()def audit_answer(answer, meta, differential=None):    source_text = _norm(" ".join(h["text"] for h in meta))    compact = source_text.replace(" ", "")    problems = []    if not re.search(r"\[S\d+\]", answer):        problems.append("no citation markers at all")    for m in NUM_UNIT_RE.finditer(answer):        frag = _norm(m.group(0))        if frag not in source_text and frag.replace(" ", "") not in compact:            problems.append(f"unsupported quantity: {m.group(0).strip()!r}")    for m in CHEM_RE.finditer(answer):        if m.group(0).lower() not in source_text:            problems.append(f"unsupported chemical: {m.group(0)!r}")    for m in re.finditer(r"\[S(\d+)\]", answer):        if not (1 <= int(m.group(1)) <= len(meta)):            problems.append(f"citation [S{m.group(1)}] does not exist")    for m in IMAGE_CLAIM_RE.finditer(answer):        problems.append(f"claims to have seen the image: {m.group(0)!r}")    if differential is not None:        sec = re.search(r"##\s*What else it could look like(.+?)(?=\n##|\Z)",                        answer, re.S | re.I)        if sec:            allowed = {pretty(d).lower() for d in differential}            for m in re.finditer(r"^\s*[-*]\s*\*\*(.+?)\*\*", sec.group(1), re.M):                named = m.group(1).strip().rstrip(":").lower()                if named and not any(a in named or named in a for a in allowed):                    problems.append(f"differential names an unretrieved disease: {m.group(1)!r}")    return (not problems), problems# ---- self-tests ---------------------------------------------------------_m = [{"text": "Mancozeb and Captan are protectants.", "doc": "t.pdf", "page": 1}]ok, p = audit_answer("Use Mancozeb as a protectant [S1].", _m)assert ok, pprint("PASS grounded")ok, p = audit_answer("The image shows brown concentric rings [S1].", _m)assert not ok and any("seen the image" in x for x in p), pprint("BLOCK image claim :", p)ok, p = audit_answer("Apply azoxystrobin at 2.5 g per litre [S1].", _m)assert not ok and any("azoxystrobin" in x for x in p), pprint("BLOCK invented chem+dose")_d = {"tomato_target_spot": []}bad = ("## What else it could look like\n"       "- **Tomato mosaic virus:** it mottles the leaf [S1].\n")ok, p = audit_answer(bad, _m, differential=_d)assert not ok and any("unretrieved disease" in x for x in p), pprint("BLOCK invented differential:", [x for x in p if "unretrieved" in x])

## 8. Generation, extractive fallback, and the advisory entry point

In [ ]:
_last = [0.0]def ask_llm(user, system):    for attempt in range(MAX_RETRIES):        w = REQ_PAUSE - (time.time() - _last[0])        if w > 0:            time.sleep(w)        _last[0] = time.time()        r = requests.post(f"{BASE_URL}/chat/completions", timeout=180,            headers={"Authorization": f"Bearer {API_KEY}",                     "Content-Type": "application/json"},            json={"model": LLM_MODEL, "temperature": LLM_TEMP,                  "messages": [{"role": "system", "content": system},                               {"role": "user", "content": user}]})        if r.status_code == 429:            back = float(r.headers.get("Retry-After", REQ_PAUSE * (attempt + 2)))            print(f"    rate limited, waiting {back:.0f}s"); time.sleep(back); continue        r.raise_for_status()        return r.json()["choices"][0]["message"]["content"]    raise RuntimeError("rate limited after retries")BLOCK_TITLES = {"identity": "What this disease is", "symptoms": "Typical symptoms",                "spread": "How it spreads", "treatment": "What to do now",                "prevention": "What to keep doing"}def extractive_advisory(case, blocks, differential, caveats):    L = [f"# {pretty(case['disease']).title()}", "",         f"**Confidence** {case['confidence']:.1%}  |  "         f"**Leaf area affected** {case['severity_pct']:.1f}% ({case['severity_level']})  |  "         f"**Urgency** {resolve_urgency(case)[0]}", ""]    for k, title in BLOCK_TITLES.items():        L.append(f"**{title}:**")        hits = blocks.get(k, [])        L += ([f"- {h['text']}  [{h['doc']} p.{h['page']}]" for h in hits]              or ["The sources do not state this."])        L.append("")    if differential:        L.append("**What else it could look like:**")        for other, hits in differential.items():            L.append(f"- *{pretty(other)}* — {hits[0]['text']}  "                     f"[{hits[0]['doc']} p.{hits[0]['page']}]")        L.append("")    for c in caveats:        L.append(f"> {c}")    L.append("\n> Diagnosis came from an image classifier, not a laboratory test. "             "Follow local registration and label rates for any chemical.")    return "\n".join(L).strip()def make_advisory(case):    cls = case["disease"]    if cls in HEALTHY_CLASSES:        return {**case, "tier": "healthy", "mode": "fixed", "audit_ok": True,                "text": f"# Healthy {case['crop']} leaf\n\n"                        f"No disease detected (confidence {case['confidence']:.1%}, "                        f"{case['severity_pct']:.1f}% of leaf area affected). "                        f"Continue routine monitoring and normal cultural practices."}    blocks, differential, caveats = build_evidence(cls)    tier = tier_for(cls)    if not blocks:        return {**case, "tier": tier, "mode": "none", "audit_ok": True,                "text": f"{pretty(cls).title()} is not covered by the knowledge base."}    if not LLM_OK:        return {**case, "tier": tier, "mode": "extractive", "audit_ok": True,                "audit_problems": ["no LLM configured"],                "text": extractive_advisory(case, blocks, differential, caveats)}    user, meta = build_reasoning_prompt(case, blocks, differential, caveats)    try:        answer = ask_llm(user, REASONING_SYSTEM)    except Exception as e:        return {**case, "tier": tier, "mode": "extractive", "audit_ok": True,                "audit_problems": [f"api error: {type(e).__name__}: {str(e)[:120]}"],                "text": extractive_advisory(case, blocks, differential, caveats)}    ok, problems = audit_answer(answer, meta, differential=differential)    if ok:        head = (f"**Confidence** {case['confidence']:.1%}  |  "                f"**Leaf area affected** {case['severity_pct']:.1f}% "                f"({case['severity_level']})  |  "                f"**Urgency** {resolve_urgency(case)[0]}\n")        srcs = "\n".join(f"[S{i+1}] {h['doc']} p.{h['page']}" for i, h in enumerate(meta))        return {**case, "tier": tier, "mode": "reasoned", "audit_ok": True,                "text": f"# {pretty(cls).title()}\n\n{head}\n{answer}\n\nSources:\n{srcs}"}    return {**case, "tier": tier, "mode": "extractive", "audit_ok": False,            "audit_problems": problems, "rejected_draft": answer,            "text": extractive_advisory(case, blocks, differential, caveats)}

## 9. Classify and measure — the image halfLoads notebook 01's checkpoint, preprocessing and severity modules. If they are missing,`classify_and_measure` returns the manual case from the config cell so the rest of thenotebook still runs.The OOD screens are the reason this cell can return `refused`. A 13-class softmax alwayssums to 1, so an unseen species produces a *confident wrong answer* — the Mahalanobis andleaf-presence screens are what stop an advisory being written for a photo of a hand.

In [ ]:
import importlib.util, numpy as np, torch, jsonimport cv2import torchvisionimport torchvision.transforms as transformsdef _load_module(path, name):    spec = importlib.util.spec_from_file_location(name, path)    m = importlib.util.module_from_spec(spec)    spec.loader.exec_module(m)    return mPRE = _load_module(PREPROC_PY, "nb01_preprocessing") if PREPROC_PY else NoneSEV = _load_module(SEVERITY_PY, "nb01_severity") if SEVERITY_PY else NoneRUN = json.loads(Path(RUN_CONFIG_JSON).read_text()) if RUN_CONFIG_JSON else {}# ---- everything below comes from run_config.json, not from guesses ------IMAGE_SIZE   = RUN.get("img_size", 224)MEAN         = RUN.get("mean", [0.485, 0.456, 0.406])STD          = RUN.get("std",  [0.229, 0.224, 0.225])IN_CHANNELS  = RUN.get("in_channels", 3)NUM_CLASSES  = RUN.get("num_classes", len(CLASSES))REJECTION    = RUN.get("rejection", {})CONFIDENCE_GATE = REJECTION.get("confidence_gate", RUN.get("confidence_gate", 0.80))OOD_THRESHOLD   = REJECTION.get("ood_threshold", None)VEGETATION_MIN  = REJECTION.get("vegetation_min", 0.06)VEGETATION_MAX  = REJECTION.get("vegetation_max", 0.97)REJECT_MESSAGES = REJECTION.get("messages", {    "no_leaf": "Cannot classify - no leaf detected in the image",    "unknown_leaf": "Cannot classify - this leaf type is not in the training dataset",    "low_confidence": "Cannot classify - image is unclear or leaf type not recognized",})# notebook 01 rebuilt eval_tf as Resize(1.14x) -> CenterCrop, NOT a plain Resize.# A plain Resize((224,224)) changes the aspect ratio and shifts the input distribution# away from what the model was validated on, so it is reproduced exactly here.eval_tf = transforms.Compose([    transforms.ToPILImage(),    transforms.Resize(int(IMAGE_SIZE * 1.14)),    transforms.CenterCrop(IMAGE_SIZE),    transforms.ToTensor(),    transforms.Normalize(MEAN, STD),])# ---- class order: class_index.json is authoritative --------------------if CLASS_INDEX_JSON:    class_to_id = json.loads(Path(CLASS_INDEX_JSON).read_text())    id_to_class = {int(i): n for n, i in class_to_id.items()}else:    id_to_class = {i: c for i, c in enumerate(RUN.get("classes", CLASSES))}MODEL_CLASSES = [id_to_class[i] for i in sorted(id_to_class)]# ---- OOD gaussian ------------------------------------------------------CLASS_MEANS = PRECISION = Noneif OOD_NPZ:    z = np.load(OOD_NPZ)    CLASS_MEANS, PRECISION = z["class_means"], z["precision"]    if OOD_THRESHOLD is None:        OOD_THRESHOLD = float(z["threshold"][0])    print(f"OOD gaussian: {CLASS_MEANS.shape[0]} centres in {CLASS_MEANS.shape[1]}d, "          f"threshold {OOD_THRESHOLD:.2f}")def mahalanobis_score(features):    best = None    for centre in CLASS_MEANS:        d = features - centre        dist = np.einsum("ij,jk,ik->i", d, PRECISION, d)        best = dist if best is None else np.minimum(best, dist)    return np.sqrt(np.maximum(best, 0.0))# ---- model -------------------------------------------------------------MODEL = Noneif CKPT:    try:        MODEL = torchvision.models.efficientnet_b0(weights=None)        if IN_CHANNELS != 3:            f = MODEL.features[0][0]            wide = torch.nn.Conv2d(IN_CHANNELS, f.out_channels, f.kernel_size,                                   f.stride, f.padding, bias=f.bias is not None)            MODEL.features[0][0] = wide        MODEL.classifier[1] = torch.nn.Linear(MODEL.classifier[1].in_features, NUM_CLASSES)        ck = torch.load(CKPT, map_location="cpu", weights_only=False)        state = ck.get("model_state_dict", ck.get("state_dict", ck))        missing, unexpected = MODEL.load_state_dict(state, strict=False)        assert not missing, f"checkpoint is missing weights for: {missing[:6]}"        MODEL.eval()        print(f"model loaded | {NUM_CLASSES} classes | {IN_CHANNELS}ch | "              f"conf gate {CONFIDENCE_GATE} | img {IMAGE_SIZE}")    except Exception as e:        print("could not load the checkpoint:", type(e).__name__, str(e)[:250])        MODEL = NoneCNN_READY = MODEL is not None and SEV is not Noneprint("CNN pipeline:", "READY" if CNN_READY else f"NOT READY -> MANUAL mode ({MANUAL_CLASS})")@torch.no_grad()def classify_and_measure(image_path=None):    """Faithful re-implementation of notebook 01's screen_and_classify + assess_severity.    Returns a case dict, or {'refused': True, ...}."""    if not CNN_READY:        lvl = SEV.severity_level(MANUAL_SEVERITY) if SEV else "Moderate"        return {"disease": MANUAL_CLASS, "crop": MANUAL_CLASS.split("_")[0],                "confidence": MANUAL_CONF, "severity_pct": MANUAL_SEVERITY,                "severity_level": lvl, "image": str(image_path or "(manual)"),                "source": "MANUAL"}    bgr = cv2.imread(str(image_path))    if bgr is None:        return {"refused": True, "reason": "unreadable",                "detail": f"cv2 could not read {image_path}", "image": str(image_path)}    # analyse the RAW image, exactly as notebook 01 does, so severity keeps its meaning    analysis = SEV.analyse_leaf(bgr)    leaf_fraction = float(analysis["leaf_fraction"])    # ---- screen 1: is there a leaf at all -----------------------------    if not (VEGETATION_MIN <= leaf_fraction <= VEGETATION_MAX):        return {"refused": True, "reason": "no_leaf",                "detail": f"leaf covers {leaf_fraction:.1%} of the frame",                "message": REJECT_MESSAGES["no_leaf"], "image": str(image_path)}    prepared = PRE.preprocess_image(bgr) if PRE else bgr    tensor = eval_tf(cv2.cvtColor(prepared, cv2.COLOR_BGR2RGB))    if IN_CHANNELS == 4:        m = cv2.resize(analysis["lesion_mask"], (tensor.shape[2], tensor.shape[1]),                       interpolation=cv2.INTER_NEAREST)        tensor = torch.cat([tensor,                            torch.from_numpy(m.astype(np.float32) / 255.0).unsqueeze(0)], 0)    batch  = tensor.unsqueeze(0)    pooled = torch.flatten(MODEL.avgpool(MODEL.features(batch)), 1)    probs  = torch.softmax(MODEL.classifier(pooled), 1)[0].numpy()    cid    = int(probs.argmax())    conf   = float(probs[cid])    # ---- screen 2: Mahalanobis distance in the 1280-d feature space ----    dist = None    if CLASS_MEANS is not None:        dist = float(mahalanobis_score(pooled.numpy())[0])        if OOD_THRESHOLD is not None and dist > OOD_THRESHOLD:            return {"refused": True, "reason": "unknown_leaf",                    "detail": f"distance {dist:.1f} / threshold {OOD_THRESHOLD:.1f}",                    "message": REJECT_MESSAGES["unknown_leaf"], "image": str(image_path)}    # ---- screen 3: confidence gate ------------------------------------    if conf < CONFIDENCE_GATE:        return {"refused": True, "reason": "low_confidence",                "detail": f"confidence {conf:.2f} / threshold {CONFIDENCE_GATE:.2f}",                "message": REJECT_MESSAGES["low_confidence"], "image": str(image_path)}    sev = SEV.assess_severity(bgr, analysis)    pct = sev["severity_percent"]    return {"disease": id_to_class[cid], "crop": id_to_class[cid].split("_")[0],            "confidence": conf,            "severity_pct": float(pct) if pct is not None else float("nan"),            "severity_level": sev["severity_level"],            "lesion_count": sev["lesion_count"],            "leaf_fraction": sev["leaf_fraction"],            "mask_ok": sev["mask_ok"],            "ood_distance": dist,            "image": str(image_path), "source": "CNN"}

## 10. One image, end to end

In [ ]:
sample = Nonefor root in IMAGE_ROOTS:    hits = list(root.rglob("*.JPG"))[:1] or list(root.rglob("*.jpg"))[:1]    if hits:        sample = hits[0]; breakprint("sample image:", sample)case = classify_and_measure(sample)if case.get("refused"):    print("REFUSED:", case["reason"], "-", case["detail"])else:    print(f"{case['disease']}  conf {case['confidence']:.1%}  "          f"severity {case['severity_pct']:.1f}% ({case['severity_level']})  "          f"[{case['source']}]")    adv = make_advisory(case)    print(f"\ntier {adv['tier']} | mode {adv['mode']} | audit {adv['audit_ok']}")    if adv.get("audit_problems"):        print("problems:", adv["audit_problems"])    print("-" * 80)    print(adv["text"])

## 11. Batch over a sample of the test setKeep `PER_CLASS` small on the free tier — one API call per image, spaced by `REQ_PAUSE`.Thirteen images at 6 s apart is about 80 seconds.

In [ ]:
PER_CLASS = 1picks = []for root in IMAGE_ROOTS:    for cls_dir in sorted({p.parent for p in root.rglob("*.JPG")} |                          {p.parent for p in root.rglob("*.jpg")}):        imgs = sorted(list(cls_dir.glob("*.JPG")) + list(cls_dir.glob("*.jpg")))[:PER_CLASS]        picks.extend(imgs)print(f"{len(picks)} images selected")rows = []for img in picks:    c = classify_and_measure(img)    if c.get("refused"):        rows.append({"image": img.name, "class": "-", "mode": "refused",                     "reason": c["reason"], "audit_ok": True, "problems": ""})        print(f"{img.name[:40]:40s} REFUSED ({c['reason']})")        continue    a = make_advisory(c)    safe = a["disease"] + "__" + Path(a["image"]).stem    (OUT / f"{safe}.md").write_text(a["text"], encoding="utf-8")    if not a["audit_ok"]:        (OUT / f"{safe}.REJECTED.txt").write_text(a.get("rejected_draft", ""),                                                  encoding="utf-8")    rows.append({"image": Path(a["image"]).name, "class": a["disease"], "tier": a["tier"],                 "mode": a["mode"], "conf": round(a["confidence"], 4),                 "severity_pct": round(a["severity_pct"], 2), "band": a["severity_level"],                 "audit_ok": a["audit_ok"],                 "problems": "; ".join(a.get("audit_problems", []))[:90]})    print(f"{Path(a['image']).name[:40]:40s} {a['disease']:30s} {a['mode']:11s} "          f"audit={a['audit_ok']}")df = pd.DataFrame(rows)df.to_csv(OUT / "advisory_evaluation.csv", index=False)print("\n" + df["mode"].value_counts().to_string())if "audit_ok" in df and (~df["audit_ok"]).any():    print("\naudit failures:")    print(df[~df["audit_ok"]][["image", "class", "problems"]].to_string(index=False))

---## Reading the output* `mode = reasoned` — the LLM wrote it and every audit check passed.* `mode = extractive` — the API was down, or the audit caught something. Check  `problems`, and read the `.REJECTED.txt` next to the advisory.* `mode = refused` — the OOD screens rejected the image before any retrieval happened.  That is the system working, not failing.The audit failure worth watching for is `claims to have seen the image`. If it appearsoften, the model is ignoring rule 1 — tighten it by moving the ban into the first line ofthe user prompt as well as the system prompt, since some models weight the lastinstruction most heavily.